<div style="background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%); padding: 40px; border-radius: 12px; text-align: center; color: white; margin-bottom: 20px;">
  <h1 style="color: #e94560; font-size: 2.2em; margin-bottom: 10px;">Pruebas Reproducibles de OPUS-MT</h1>
  <h2 style="color: #ffffff; font-size: 1.4em;">Evaluación de 24 frases de español a inglés con un modelo preentrenado para verificar sus métricas.</h2>
  <hr style="border-color: #e94560; margin: 20px 0;">
  <p style="color: #ccc;"><b>Asignatura:</b> Procesamiento de Datos Secuenciales con Deep Learning</p>
  <p style="color: #ccc;"><b>Programa:</b> Maestría en Inteligencia Artificial y Ciencia de Datos</p>
  <p style="color: #ccc;"><b>Profesor:</b> Natali Johana Velandia — Universidad Autónoma de Occidente</p>
  <p style="color: #ccc;"><b>Estudiantes:</b> Alejandro Meneses, Marko David García, Henry Rosenstiehl.</p>
  <p style="color: #ccc;"><b>Son pruebas propias del proyecto, no experimentos del paper. Hay una referencia por frase y tres repeticiones para medir tiempos.</p>
</div>

In [15]:
%pip install torch==2.11.0 transformers==5.17.0 sentencepiece==0.2.2 sacremoses==0.2.0 sacrebleu==2.5.1 numpy pandas plotly==7.1.0

Note: you may need to restart the kernel to use updated packages.


## Configuración

Fijamos la revisión del modelo y usamos float32, seis hilos y cuatro haces de búsqueda. Los tiempos dependen del equipo. Las métricas se calculan a partir de las traducciones generadas durante esta ejecución.

In [16]:
import importlib.metadata
import json
import platform
import time
from datetime import datetime, timezone
from pathlib import Path

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sacrebleu.metrics import BLEU, CHRF
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MODELO = "Helsinki-NLP/opus-mt-es-en"
REVISION = "c96e2c5399ebfae4fc43d9669556b9afa74bb69d"
REPETICIONES = 3
SEMILLA = 7
GENERACION = {"max_new_tokens": 256, "num_beams": 4, "early_stopping": True}

torch.set_num_threads(6)
torch.manual_seed(SEMILLA)
paquetes = ["torch", "transformers", "sentencepiece", "sacremoses", "sacrebleu", "numpy", "pandas", "plotly"]
versiones = {nombre: importlib.metadata.version(nombre) for nombre in paquetes}
display(pd.Series(versiones, name="Versión"))

torch            2.11.0+cu128
transformers           5.17.0
sentencepiece           0.2.2
sacremoses              0.2.0
sacrebleu               2.5.1
numpy                   2.5.2
pandas                  3.0.5
plotly                  7.1.0
Name: Versión, dtype: str

## Datos de prueba

Son seis grupos con cuatro frases cada uno. Las referencias se prepararon antes de ver las traducciones. Esta muestra manual sirve para explorar errores, no para estimar la calidad general del modelo.

In [ ]:
CASOS = [['cotidiano', 'El gato duerme sobre la silla.', 'The cat sleeps on the chair.'],
 ['cotidiano',
  'Mañana vamos a visitar a mi abuela.',
  'Tomorrow we are going to visit my grandmother.'],
 ['cotidiano', '¿Puedes cerrar la ventana, por favor?', 'Can you close the window, please?'],
 ['cotidiano', 'No encontré las llaves en la cocina.', 'I did not find the keys in the kitchen.'],
 ['tecnico',
  'El modelo utiliza ocho cabezas de atención.',
  'The model uses eight attention heads.'],
 ['tecnico',
  'Los datos se dividen en entrenamiento, validación y prueba.',
  'The data is split into training, validation, and test sets.'],
 ['tecnico',
  'La red predice el siguiente token a partir del contexto.',
  'The network predicts the next token from the context.'],
 ['tecnico',
  'El servidor devolvió un error al cargar los pesos.',
  'The server returned an error while loading the weights.'],
 ['ambiguedad', 'Me senté en el banco del parque.', 'I sat on the park bench.'],
 ['ambiguedad', 'El banco aprobó el préstamo ayer.', 'The bank approved the loan yesterday.'],
 ['ambiguedad', 'La vela iluminó la habitación.', 'The candle lit up the room.'],
 ['ambiguedad', 'El viento llenó la vela del barco.', 'The wind filled the sail of the boat.'],
 ['modismos', 'Me estás tomando el pelo.', 'You are pulling my leg.'],
 ['modismos', 'Este examen fue pan comido.', 'This exam was a piece of cake.'],
 ['modismos', 'No hay que tirar la toalla todavía.', 'We should not throw in the towel yet.'],
 ['modismos', 'Se me fue el santo al cielo.', 'I lost my train of thought.'],
 ['entidades', 'Alejandro vive en Cali, Colombia.', 'Alejandro lives in Cali, Colombia.'],
 ['entidades', 'La reunión empieza a las 3:30 p. m.', 'The meeting starts at 3:30 p.m.'],
 ['entidades',
  'El informe contiene 25 tablas y 8 gráficos.',
  'The report contains 25 tables and 8 charts.'],
 ['entidades',
  'El archivo se llama resultados_2026.csv.',
  'The file is called resultados_2026.csv.'],
 ['estructura',
  'Aunque estaba lloviendo, salimos a caminar.',
  'Although it was raining, we went for a walk.'],
 ['estructura',
  'Si hubiera estudiado más, habría aprobado el examen.',
  'If I had studied more, I would have passed the exam.'],
 ['estructura',
  'La investigadora que presentó el artículo respondió las preguntas.',
  'The researcher who presented the paper answered the questions.'],
 ['estructura',
  'El sistema no debe eliminar los datos que aún no se han guardado.',
  'The system must not delete data that has not yet been saved.']]

corpus = pd.DataFrame(CASOS, columns=["grupo", "fuente", "referencia"])
corpus.insert(0, "id", range(1, len(corpus) + 1))

assert len(corpus) == 24 and corpus.groupby("grupo").size().eq(4).all()
assert corpus[["fuente", "referencia"]].map(lambda s: isinstance(s, str) and bool(s.strip())).all().all()

display(corpus)

,id,grupo,fuente,referencia
0,1,cotidiano,El gato duerme sobre la silla.,The cat sleeps on the chair.
1,2,cotidiano,Mañana vamos a visitar a mi abuela.,Tomorrow we are going to visit my grandmother.
2,3,cotidiano,"¿Puedes cerrar la ventana, por favor?","Can you close the window, please?"
3,4,cotidiano,No encontré las llaves en la cocina.,I did not find the keys in the kitchen.
4,5,tecnico,El modelo utiliza ocho cabezas de atención.,The model uses eight attention heads.
5,6,tecnico,"Los datos se dividen en entrenamiento, validac...","The data is split into training, validation, a..."
6,7,tecnico,La red predice el siguiente token a partir del...,The network predicts the next token from the c...
7,8,tecnico,El servidor devolvió un error al cargar los pe...,The server returned an error while loading the...
8,9,ambiguedad,Me senté en el banco del parque.,I sat on the park bench.
9,10,ambiguedad,El banco aprobó el préstamo ayer.,The bank approved the loan yesterday.


## Cargar y traducir

`from_pretrained` carga los pesos publicados. `eval()` desactiva dropout y `no_grad()` evita calcular gradientes.

La función tokeniza el texto, genera la traducción y devuelve el texto y sus IDs. La configuración de generación se define en `GENERACION`.

In [18]:
def cargar_modelo(nombre, revision):
    """Carga el tokenizador y el modelo en CPU."""
    tokenizador = AutoTokenizer.from_pretrained(nombre, revision=revision)
    modelo = AutoModelForSeq2SeqLM.from_pretrained(
        nombre, revision=revision, attn_implementation="eager"
    )
    modelo = modelo.to(device="cpu", dtype=torch.float32).eval()
    return tokenizador, modelo


tokenizador, modelo = cargar_modelo(MODELO, REVISION)

Loading weights: 100%|██████████| 258/258 [00:00<00:00, 17759.64it/s]


In [19]:
def traducir(texto, tokenizador, modelo):
    """Traduce una frase y conserva los IDs para analizarla."""
    entrada = tokenizador(texto, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        salida = modelo.generate(**entrada, **GENERACION)
    return {
        "salida": tokenizador.decode(salida[0], skip_special_tokens=True),
        "entrada_ids": entrada["input_ids"][0].tolist(),
        "salida_ids": salida[0].tolist(),
    }


def evaluar_corpus(corpus, tokenizador, modelo, repeticiones=3):
    """Traduce cada frase varias veces y registra salidas y tiempos."""
    if corpus.empty or repeticiones < 1:
        raise ValueError("Se necesita al menos una frase y una repetición.")
    traducir("Hola mundo.", tokenizador, modelo)  # Calentamiento fuera de la medición.
    registros = []
    for caso in corpus.to_dict("records"):
        tiempos = []
        traducciones = []
        for _ in range(repeticiones):
            inicio = time.perf_counter()
            traduccion = traducir(caso["fuente"], tokenizador, modelo)
            tiempos.append(time.perf_counter() - inicio)
            traducciones.append(traduccion["salida"])
        registros.append({
            **caso, **traduccion,
            "salidas_repeticiones": traducciones,
            "estable": len(set(traducciones)) == 1,
            "tiempos_s": tiempos,
            "mediana_s": float(np.median(tiempos)),
            "tokens_generados": len(traduccion["salida_ids"]) - 1,
        })
    return pd.DataFrame(registros)

In [20]:
resultados = evaluar_corpus(corpus, tokenizador, modelo, REPETICIONES)

print("Traducciones estables:", resultados["estable"].sum(), "de", len(resultados))
display(resultados[["id", "grupo", "fuente", "referencia", "salida"]])

[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface

Traducciones estables: 24 de 24


,id,grupo,fuente,referencia,salida
0,1,cotidiano,El gato duerme sobre la silla.,The cat sleeps on the chair.,The cat sleeps on the chair.
1,2,cotidiano,Mañana vamos a visitar a mi abuela.,Tomorrow we are going to visit my grandmother.,Tomorrow we're going to visit my grandmother.
2,3,cotidiano,"¿Puedes cerrar la ventana, por favor?","Can you close the window, please?","Can you close the window, please?"
3,4,cotidiano,No encontré las llaves en la cocina.,I did not find the keys in the kitchen.,I didn't find the keys in the kitchen.
4,5,tecnico,El modelo utiliza ocho cabezas de atención.,The model uses eight attention heads.,The model uses eight heads of attention.
5,6,tecnico,"Los datos se dividen en entrenamiento, validac...","The data is split into training, validation, a...","The data is divided into training, validation ..."
6,7,tecnico,La red predice el siguiente token a partir del...,The network predicts the next token from the c...,The network predicts the next token from the c...
7,8,tecnico,El servidor devolvió un error al cargar los pe...,The server returned an error while loading the...,The server returned an error when loading the ...
8,9,ambiguedad,Me senté en el banco del parque.,I sat on the park bench.,I sat at the park bench.
9,10,ambiguedad,El banco aprobó el préstamo ayer.,The bank approved the loan yesterday.,The bank approved the loan yesterday.


## Evaluar los resultados

**BLEU** compara n-gramas de palabras mientras que **chrF2** compara n-gramas de caracteres. Son puntajes de corpus y no porcentajes de aciertos.
Calculamos los tiempos sobre las 72 llamadas incluyendo tokenización, generación y decodificación pero no la carga del modelo. Los tokens generados incluyen `<EOS>` y excluyen el token inicial.

In [ ]:
def calcular_metricas(resultados):
    """Calcula BLEU y chrF2 sobre todas las frases recibidas."""
    traducciones = resultados["salida"].tolist()
    referencias = [resultados["referencia"].tolist()]
    return {
        "BLEU": BLEU().corpus_score(traducciones, referencias).score,
        "chrF2": CHRF().corpus_score(traducciones, referencias).score,
    }


def resumir_por_grupo(resultados):
    resumen = []
    for grupo, frases in resultados.groupby("grupo", sort=False):
        resumen.append({"grupo": grupo, "n": len(frases), **calcular_metricas(frases)})
    return pd.DataFrame(resumen)


metricas = calcular_metricas(resultados)
por_grupo = resumir_por_grupo(resultados)

display(pd.Series(metricas, name="Puntaje").round(3))
display(por_grupo.round(3))

tiempos = np.concatenate(resultados["tiempos_s"].to_list())
rendimiento = {
    "mediana_s": float(np.median(tiempos)),
    "p95_s": float(np.percentile(tiempos, 95)),
    "tokens_s": float(resultados["tokens_generados"].sum() * REPETICIONES / tiempos.sum()),
}

display(pd.Series(rendimiento, name="Rendimiento actual"))

BLEU     70.994
chrF2    82.613
Name: Puntaje, dtype: float64

,grupo,n,BLEU,chrF2
0,cotidiano,4,80.369,93.479
1,tecnico,4,61.529,83.561
2,ambiguedad,4,81.479,89.956
3,modismos,4,39.827,45.376
4,entidades,4,85.034,90.450
5,estructura,4,71.831,83.721


mediana_s     0.290259
p95_s         0.408666
tokens_s     35.651725
Name: Rendimiento actual, dtype: float64

### Gráficos e intervalos

El bootstrap remuestrea estas 24 frases 1.000 veces, con semilla 7. El intervalo del 95 % describe la variación de esta muestra y no elimina su sesgo de selección. Cada grupo tiene apenas cuatro frases.

In [ ]:
def calcular_intervalos(resultados, remuestreos=1000, semilla=7):
    """Obtiene intervalos percentiles mediante bootstrap de frases."""
    
    generador = np.random.default_rng(semilla)
    muestras = []
    
    for _ in range(remuestreos):
        indices = generador.integers(0, len(resultados), len(resultados))
        muestras.append(calcular_metricas(resultados.iloc[indices]))
    intervalos = pd.DataFrame(muestras).quantile([0.025, 0.975]).T
    intervalos.columns = ["Límite inferior", "Límite superior"]
    
    return intervalos


intervalos = calcular_intervalos(resultados, semilla=SEMILLA)
display(intervalos.round(3))

,Límite inferior,Límite superior
BLEU,61.897,79.921
chrF2,76.223,88.470


### Gráficas interactivas de calidad y tiempo

A la izquierda se comparan BLEU y chrF2 por grupo. A la derecha hay un punto por frase, con los tokens generados en el eje horizontal y la mediana de sus tres tiempos en el vertical. Se incluye el token de fin de secuencia y se excluye el token inicial.

In [ ]:
def graficar_metricas(resultados, metricas_por_grupo):
    """Muestra calidad y tiempos con valores al pasar el cursor."""
    
    nombres = {"cotidiano": "Cotidiano", "tecnico": "Técnico", "ambiguedad": "Ambigüedad",
               "modismos": "Modismos", "entidades": "Entidades", "estructura": "Estructura"}
    
    grupos = metricas_por_grupo["grupo"].replace(nombres).tolist()
    
    figura = make_subplots(rows=1, cols=2, horizontal_spacing=0.16,
                           subplot_titles=("Calidad por grupo", "Tiempo por frase"))
    
    for metrica, color in [("BLEU", "#245F82"), ("chrF2", "#48A29C")]:
        figura.add_trace(go.Bar(
            x=metricas_por_grupo[metrica].tolist(), y=grupos, orientation="h",
            name=metrica, marker_color=color,
            hovertemplate="%{y}<br>Puntaje: %{x:.2f}<extra>" + metrica + "</extra>",
        ), row=1, col=1)
    
    figura.add_trace(go.Scatter(
        x=resultados["tokens_generados"].tolist(), y=resultados["mediana_s"].tolist(),
        mode="markers", marker=dict(color="#245F82", size=10), showlegend=False,
        customdata=resultados[["id", "fuente", "salida"]].values.tolist(),
        hovertemplate="Caso %{customdata[0]}<br>%{customdata[1]}<br>%{customdata[2]}"
                      "<br>Tokens: %{x}<br>Mediana: %{y:.3f} s<extra></extra>",
    ), row=1, col=2)
    
    figura.update_xaxes(title_text="Puntaje de 0 a 100", range=[0, 105], row=1, col=1)
    figura.update_yaxes(autorange="reversed", row=1, col=1)
    figura.update_xaxes(title_text="Tokens generados, incluye EOS", dtick=1, row=1, col=2)
    figura.update_yaxes(title_text="Mediana en segundos", row=1, col=2)
    figura.update_layout(template="plotly_white", barmode="group", height=500,
                          margin=dict(l=100, r=30, t=80, b=70), legend=dict(orientation="h"))
    return figura


figura_metricas = graficar_metricas(resultados, por_grupo)
figura_metricas.show()

## Análisis de errores

Después de ejecutar el modelo, se compara las traducciones con las fuentes y referencias, se registra los errores sin asumir de antemano cuál será la salida.

| Casos | Qué revisar |
|---|---|
| 5–8 | Terminología técnica y naturalidad de la traducción |
| 9–12 | Sentido de palabras ambiguas como banco y vela |
| 13–16 | Conservación del significado de los modismos |
| 17–20 | Nombres, cifras, horas e identificadores |
| 21–24 | Relaciones entre cláusulas, negación y fuerza de las obligaciones |

Una traducción puede ser válida aunque difiera de la referencia. BLEU y chrF no reemplazan la revisión del significado. Estas observaciones son solo una revisión exploratoria.

In [24]:
display(resultados[["id", "grupo", "fuente", "referencia", "salida"]])

,id,grupo,fuente,referencia,salida
0,1,cotidiano,El gato duerme sobre la silla.,The cat sleeps on the chair.,The cat sleeps on the chair.
1,2,cotidiano,Mañana vamos a visitar a mi abuela.,Tomorrow we are going to visit my grandmother.,Tomorrow we're going to visit my grandmother.
2,3,cotidiano,"¿Puedes cerrar la ventana, por favor?","Can you close the window, please?","Can you close the window, please?"
3,4,cotidiano,No encontré las llaves en la cocina.,I did not find the keys in the kitchen.,I didn't find the keys in the kitchen.
4,5,tecnico,El modelo utiliza ocho cabezas de atención.,The model uses eight attention heads.,The model uses eight heads of attention.
5,6,tecnico,"Los datos se dividen en entrenamiento, validac...","The data is split into training, validation, a...","The data is divided into training, validation ..."
6,7,tecnico,La red predice el siguiente token a partir del...,The network predicts the next token from the c...,The network predicts the next token from the c...
7,8,tecnico,El servidor devolvió un error al cargar los pe...,The server returned an error while loading the...,The server returned an error when loading the ...
8,9,ambiguedad,Me senté en el banco del parque.,I sat on the park bench.,I sat at the park bench.
9,10,ambiguedad,El banco aprobó el préstamo ayer.,The bank approved the loan yesterday.,The bank approved the loan yesterday.


## Atención real y tensores Q, K y V

Recalculamos la atención sobre la traducción final de la primera frase usando un forward adicional. **No reconstruye el historial del beam search y no demuestra por sí sola la causa de una decisión.**

En la última capa de atención cruzada, Q viene de los estados del decoder que recibe ese subbloque. K y V vienen de los estados finales del encoder. Las tres proyecciones tienen pesos aprendidos. Después se separan en ocho cabezas de dimensión 64.

La operación es $A=\mathrm{softmax}(QK^T/\sqrt{d_k})$ y el contexto es $AV$. Este ejemplo no tiene padding en la entrada, por lo que no necesita aplicar una máscara de padding al reconstruir A. Las filas corresponden al token que se predice, desplazado una posición respecto al prefijo de entrada del decoder.

In [25]:
def capturar_proyeccion(nombre, proyecciones):
    """Crea un hook que guarda la salida de una proyección."""
    def guardar(modulo, entradas, salida):
        proyecciones[nombre] = salida.detach().cpu()
    
    return guardar


def separar_cabezas(tensor, numero_cabezas):
    lote, longitud, dimension = tensor.shape
    
    return tensor.reshape(lote, longitud, numero_cabezas, dimension // numero_cabezas).transpose(1, 2)


def extraer_atencion(traduccion, modelo):
    """Recalcula la atención y captura Q, K y V de la última capa."""
    proyecciones = {}
    atencion_cruzada = modelo.model.decoder.layers[-1].encoder_attn
    hooks = []
    try:
        for nombre in ["q_proj", "k_proj", "v_proj"]:
            proyeccion = getattr(atencion_cruzada, nombre)
            hooks.append(proyeccion.register_forward_hook(capturar_proyeccion(nombre, proyecciones)))
        with torch.no_grad():
            salida = modelo(
                input_ids=torch.tensor([traduccion["entrada_ids"]]),
                decoder_input_ids=torch.tensor([traduccion["salida_ids"][:-1]]),
                output_attentions=True, use_cache=False,
            )
    finally:
        for hook in hooks:
            hook.remove()
    pesos = torch.stack([capa[0].detach().cpu() for capa in salida.cross_attentions])
    cabezas = modelo.config.decoder_attention_heads
    consulta = separar_cabezas(proyecciones["q_proj"], cabezas)
    clave = separar_cabezas(proyecciones["k_proj"], cabezas)
    valor = separar_cabezas(proyecciones["v_proj"], cabezas)
    return pesos, consulta, clave, valor


def graficar_atencion(pesos, traduccion, tokenizador):
    """Permite explorar el peso de atención entre cada par de tokens."""
    tokens_entrada = tokenizador.convert_ids_to_tokens(traduccion["entrada_ids"])
    tokens_salida = tokenizador.convert_ids_to_tokens(traduccion["salida_ids"][1:])
    # Incluimos la posición para distinguir tokens repetidos.
    etiquetas_entrada = [f"{pos}: {token}" for pos, token in enumerate(tokens_entrada)]
    etiquetas_salida = [f"{pos}: {token}" for pos, token in enumerate(tokens_salida)]
    figura = go.Figure(go.Heatmap(
        z=pesos[-1].mean(0).tolist(), x=etiquetas_entrada, y=etiquetas_salida,
        colorscale="Blues", zmin=0, zmax=1, colorbar=dict(title="Peso"),
        hovertemplate="Origen: %{x}<br>Predicción: %{y}<br>Atención: %{z:.4f}<extra></extra>",
    ))
    figura.update_layout(template="plotly_white", height=550,
                          title="Atención: última capa, promedio de cabezas",
                          xaxis_title="Tokens de origen", yaxis_title="Token predicho")
    figura.update_yaxes(autorange="reversed")
    return figura

ejemplo = traducir(corpus.iloc[0]["fuente"], tokenizador, modelo)
atencion, consulta, clave, valor = extraer_atencion(ejemplo, modelo)
pesos_reconstruidos = torch.softmax(consulta @ clave.transpose(-1, -2) / np.sqrt(consulta.shape[-1]), dim=-1)

assert torch.allclose(pesos_reconstruidos[0], atencion[-1], atol=1e-6)
assert torch.allclose(atencion.sum(-1), torch.ones_like(atencion.sum(-1)), atol=1e-6)

print("Q:", tuple(consulta.shape), "K:", tuple(clave.shape), "V:", tuple(valor.shape))

figura_atencion = graficar_atencion(atencion, ejemplo, tokenizador)
figura_atencion.show()

[transformers] Both `max_new_tokens` (=256) and `max_length`(=512) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: (1, 8, 9, 64) K: (1, 8, 8, 64) V: (1, 8, 8, 64)


## Límite de entrada

Comprobamos el truncamiento del tokenizador con una entrada larga. Este chequeo no traduce el texto largo ni mide su calidad. Perder tokens de entrada puede dejar información fuera de la traducción.

In [ ]:
texto_largo = "Esta oración debe conservarse completa. " * 150
ids_completos = tokenizador(texto_largo, truncation=False)["input_ids"]
ids_recortados = tokenizador(texto_largo, truncation=True)["input_ids"]

print("Tokens originales:", len(ids_completos))
print("Tokens después del truncamiento:", len(ids_recortados))

assert len(ids_recortados) <= tokenizador.model_max_length

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1051 > 512). Running this sequence through the model will result in indexing errors


Tokens originales: 1051
Tokens después del truncamiento: 512


## Guardar los resultados

Se crea una carpeta nueva con las traducciones en CSV, el reporte en JSON, los tensores de atención y las dos gráficas interactivas en HTML. La barra de herramientas de cada gráfica permite descargar una imagen PNG. El JSON incluye las fuentes y referencias, las versiones y la configuración. No sobrescribe pruebas anteriores. En Colab puedes descargar estos archivos desde el panel lateral.

In [ ]:
def guardar_resultados(resultados, resumen, carpeta="ejecuciones_opus"):
    """Crea una carpeta nueva y guarda la tabla y los metadatos."""
  
    fecha = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    destino = Path(carpeta) / fecha
    destino.mkdir(parents=True, exist_ok=False)
    
    reporte = {**resumen, "casos": resultados.to_dict("records")}
    
    (destino / "resultados.json").write_text(
        json.dumps(reporte, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    resultados.to_csv(destino / "traducciones.csv", index=False, encoding="utf-8-sig")

    return destino


resumen = {
    "modelo": MODELO, "revision": REVISION, "versiones": versiones,
    "python": platform.python_version(), "plataforma": platform.platform(),
    "device": "cpu", "dtype": "float32", "threads": torch.get_num_threads(),
    "generacion": GENERACION, "repeticiones": REPETICIONES, "calentamiento": 1,
    "metricas": metricas, "rendimiento": rendimiento,
    "intervalos": intervalos.to_dict("index"), "bootstrap_semilla": SEMILLA,
    "bootstrap_remuestreos": 1000, "grupos": por_grupo.to_dict("records"),
    "config_metricas": {"BLEU": "13a, case:mixed, smooth:exp", "chrF2": "char_order:6, word_order:0, beta:2"},
}

destino = guardar_resultados(resultados, resumen)
np.savez_compressed(destino / "atencion.npz", atencion=atencion.numpy(),
                    Q=consulta.numpy(), K=clave.numpy(), V=valor.numpy())

figura_metricas.write_html(destino / "metricas.html", include_plotlyjs=True)
figura_atencion.write_html(destino / "atencion.html", include_plotlyjs=True)

print("Archivos guardados en:", destino.resolve())

Archivos guardados en: /home/alejo/projects/DataScience/datos_secuenciales/proyecto_final/ejecuciones_opus/20260930T152046_693065Z


## Qué podemos concluir

Este ejercicio permite comprobar salidas, reproducir métricas y observar problemas concretos de traducción. No permite afirmar que el modelo alcance este desempeño sobre cualquier corpus ni compararlo directamente con los conjuntos de evaluación del artículo.

Para ampliar el experimento se puede usar un corpus externo, añadir varias referencias, separar categorías con más ejemplos y hacer otra evaluación humana. COMET ya ahce algo similar y aparece en el paper y sería una métrica adicional a las dos calculadas aquí.

### Fuentes y procedencia

- Artículo base: J. Tiedemann et al., *Democratizing Neural Machine Translation with OPUS-MT*, [arXiv:2212.01936v3](https://arxiv.org/abs/2212.01936v3).
- Modelo utilizado: [Helsinki-NLP/opus-mt-es-en](https://huggingface.co/Helsinki-NLP/opus-mt-es-en).
- Implementación y firmas de métricas: [SacreBLEU](https://github.com/mjpost/sacrebleu).